In [2]:
from pathlib import Path
from calendar import monthrange

import numpy as np
import pandas as pd


# =============================================================================
# SETTINGS
# =============================================================================

INPUT_FOLDER = Path(".")
OUTPUT_FOLDER = Path("generated_data")

START_DATE = pd.Timestamp("2023-02-01")
END_DATE = pd.Timestamp("2025-12-31")

TRANSACTIONS_PER_MONTH = 10_000
LINES_PER_TRANSACTION = 2

RANDOM_SEED = 42
DATE_FORMAT = "%d/%m/%Y"
SEPARATOR = ";"

rng = np.random.default_rng(RANDOM_SEED)


# =============================================================================
# FILE PATHS
# =============================================================================

TRANSACTIONS_FILE = INPUT_FOLDER / "transactions.csv"
TRANSACTION_LINES_FILE = INPUT_FOLDER / "transactionline.csv"
CUSTOMER_FILE = INPUT_FOLDER / "customer.csv"
ITEM_FILE = INPUT_FOLDER / "item.csv"
SUBSIDIARY_FILE = INPUT_FOLDER / "subsidiary.csv"
SALES_BUDGET_FILE = INPUT_FOLDER / "sales_budget.csv"
FX_RATE_FILE = INPUT_FOLDER / "fx_avg_rate.csv"

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


# =============================================================================
# HELPERS
# =============================================================================

def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

    return pd.read_csv(path, sep=SEPARATOR)


def save_csv(df: pd.DataFrame, filename: str) -> None:
    output_path = OUTPUT_FOLDER / filename
    df.to_csv(
        output_path,
        sep=SEPARATOR,
        index=False,
        date_format=DATE_FORMAT
    )
    print(f"Created: {output_path.resolve()} | Rows: {len(df):,}")


def random_dates(
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    count: int
) -> pd.DatetimeIndex:
    number_of_days = (end_date - start_date).days + 1
    offsets = rng.integers(0, number_of_days, size=count)

    return pd.DatetimeIndex(start_date + pd.to_timedelta(offsets, unit="D"))


def month_end_dates(
    start_date: pd.Timestamp,
    end_date: pd.Timestamp
) -> pd.DatetimeIndex:
    return pd.date_range(
        start=start_date,
        end=end_date,
        freq="ME"
    )


# =============================================================================
# LOAD SOURCE DATA
# =============================================================================

transactions = read_csv(TRANSACTIONS_FILE)
transaction_lines = read_csv(TRANSACTION_LINES_FILE)
customers = read_csv(CUSTOMER_FILE)
items = read_csv(ITEM_FILE)
subsidiaries = read_csv(SUBSIDIARY_FILE)
sales_budget = read_csv(SALES_BUDGET_FILE)
fx_rates = read_csv(FX_RATE_FILE)

for column in [
    "transaction_date",
    "transaction_last_modified_date",
    "expected_delivery_date"
]:
    transactions[column] = pd.to_datetime(
        transactions[column],
        dayfirst=True,
        errors="coerce"
    )

transaction_lines["transaction_line_last_modified_date"] = pd.to_datetime(
    transaction_lines["transaction_line_last_modified_date"],
    dayfirst=True,
    errors="coerce"
)

sales_budget["budget_date"] = pd.to_datetime(
    sales_budget["budget_date"],
    dayfirst=True,
    errors="coerce"
)


# =============================================================================
# GENERATE TRANSACTIONS AND TRANSACTION LINES
# =============================================================================

customer_ids = customers["customer_nsid"].dropna().astype(int).to_numpy()
item_ids = items["item_nsid"].dropna().astype(int).to_numpy()
business_unit_ids = subsidiaries["bu_nsid"].dropna().astype(int).to_numpy()

next_transaction_id = int(transactions["transaction_nsid"].max()) + 1
next_transaction_line_id = (
    int(transaction_lines["transaction_line_nsid"].max()) + 1
)

generated_transactions = []
generated_transaction_lines = []

monthly_periods = pd.period_range(
    START_DATE,
    END_DATE,
    freq="M"
)

status_options = {
    "Invoice": ["Fully Billed"],
    "Sales Order": [
        "Ongoing",
        "Under Discussion",
        "Closed - Won",
        "Closed - Lost"
    ],
    "Opportunity": [
        "Ongoing",
        "Under Discussion",
        "Closed - Won",
        "Closed - Lost"
    ]
}

transaction_types = np.array([
    "Invoice",
    "Sales Order",
    "Opportunity"
])

transaction_type_probabilities = np.array([
    0.17,
    0.61,
    0.22
])

for period in monthly_periods:
    month_start = period.start_time.normalize()
    month_end = period.end_time.normalize()

    row_count = TRANSACTIONS_PER_MONTH

    transaction_ids = np.arange(
        next_transaction_id,
        next_transaction_id + row_count
    )

    transaction_dates = random_dates(
        month_start,
        month_end,
        row_count
    )

    selected_types = rng.choice(
        transaction_types,
        size=row_count,
        p=transaction_type_probabilities
    )

    selected_statuses = np.array([
        rng.choice(status_options[transaction_type])
        for transaction_type in selected_types
    ])

    modified_days = rng.integers(1, 5, size=row_count)

    transaction_modified_dates = (
        transaction_dates
        + pd.to_timedelta(modified_days, unit="D")
    )

    delivery_days = rng.integers(7, 31, size=row_count)

    expected_delivery_dates = (
        transaction_dates
        + pd.to_timedelta(delivery_days, unit="D")
    )

    expected_delivery_dates = pd.Series(expected_delivery_dates)

    # Invoices generally do not require expected delivery dates
    expected_delivery_dates.loc[
        selected_types == "Invoice"
    ] = pd.NaT

    month_transactions = pd.DataFrame({
        "transaction_nsid": transaction_ids,
        "transaction_type": selected_types,
        "transaction_status": selected_statuses,
        "transaction_number": [
            f"TRAN{transaction_id:09d}"
            for transaction_id in transaction_ids
        ],
        "transaction_date": transaction_dates,
        "transaction_last_modified_date": transaction_modified_dates,
        "expected_delivery_date": expected_delivery_dates,
        "bu_nsid": rng.choice(
            business_unit_ids,
            size=row_count
        ),
        "customer_nsid": rng.choice(
            customer_ids,
            size=row_count
        )
    })

    generated_transactions.append(month_transactions)

    # -------------------------------------------------------------------------
    # Generate two lines per transaction
    # -------------------------------------------------------------------------

    line_count = row_count * LINES_PER_TRANSACTION

    repeated_transaction_ids = np.repeat(
        transaction_ids,
        LINES_PER_TRANSACTION
    )

    transaction_line_numbers = np.tile(
        np.arange(1, LINES_PER_TRANSACTION + 1),
        row_count
    )

    repeated_transaction_dates = np.repeat(
        transaction_dates.to_numpy(),
        LINES_PER_TRANSACTION
    )

    line_modified_offsets = rng.integers(
        0,
        6,
        size=line_count
    )

    line_modified_dates = (
        pd.to_datetime(repeated_transaction_dates)
        + pd.to_timedelta(line_modified_offsets, unit="D")
    )

    quantities = rng.integers(
        1,
        5_001,
        size=line_count
    )

    foreign_amounts = np.round(
        rng.lognormal(
            mean=8.0,
            sigma=1.0,
            size=line_count
        ),
        2
    )

    foreign_amounts = np.clip(
        foreign_amounts,
        50,
        250_000
    )

    month_transaction_lines = pd.DataFrame({
        "transaction_nsid": repeated_transaction_ids,
        "transaction_line_nsid": np.arange(
            next_transaction_line_id,
            next_transaction_line_id + line_count
        ),
        "quantity": quantities,
        "foreign_amount": foreign_amounts,
        "foreign_currency": "USD",
        "bu_rate": np.round(
            rng.uniform(0.85, 1.15, size=line_count),
            4
        ),
        "item_nsid": rng.choice(
            item_ids,
            size=line_count
        ),
        "transaction_line_last_modified_date": line_modified_dates
    })

    generated_transaction_lines.append(month_transaction_lines)

    next_transaction_id += row_count
    next_transaction_line_id += line_count


new_transactions = pd.concat(
    generated_transactions,
    ignore_index=True
)

new_transaction_lines = pd.concat(
    generated_transaction_lines,
    ignore_index=True
)


# =============================================================================
# GENERATE SALES BUDGETS
# =============================================================================

original_2023_budget = sales_budget[
    sales_budget["budget_year"] == 2023
].copy()

generated_budgets = []

for year in [2024, 2025]:
    annual_growth_rate = {
        2024: 1.08,
        2025: 1.16
    }[year]

    year_budget = original_2023_budget.copy()

    year_budget["budget_year"] = year

    year_budget["budget_date"] = year_budget["budget_date"].apply(
        lambda date: pd.Timestamp(
            year=year,
            month=date.month,
            day=monthrange(year, date.month)[1]
        )
    )

    year_budget["sales_amount_bu_currency"] = (
        year_budget["sales_amount_bu_currency"]
        * annual_growth_rate
        * rng.uniform(0.94, 1.06, size=len(year_budget))
    ).round(2)

    generated_budgets.append(year_budget)

new_sales_budget = pd.concat(
    [sales_budget, *generated_budgets],
    ignore_index=True
)

new_sales_budget = (
    new_sales_budget
    .drop_duplicates(
        subset=[
            "budget_version",
            "budget_date",
            "customer_name",
            "bu_code"
        ],
        keep="last"
    )
    .sort_values([
        "budget_date",
        "bu_code",
        "customer_name"
    ])
    .reset_index(drop=True)
)


# =============================================================================
# GENERATE FX AVERAGE RATES
# =============================================================================

# Source FX file is in wide format:
# original_currency;target_currency;31/01/2024;29/02/2024;...

identifier_columns = [
    "original_currency",
    "target_currency"
]

rate_columns = [
    column
    for column in fx_rates.columns
    if column not in identifier_columns
]

long_fx = fx_rates.melt(
    id_vars=identifier_columns,
    value_vars=rate_columns,
    var_name="rate_date",
    value_name="avg_rate"
)

long_fx["rate_date"] = pd.to_datetime(
    long_fx["rate_date"],
    dayfirst=True,
    errors="coerce"
)

long_fx["avg_rate"] = pd.to_numeric(
    long_fx["avg_rate"],
    errors="coerce"
)

latest_rates = (
    long_fx
    .sort_values("rate_date")
    .groupby(
        identifier_columns,
        as_index=False
    )
    .tail(1)
    .reset_index(drop=True)
)

generated_fx_records = []

fx_months = month_end_dates(
    pd.Timestamp("2024-04-01"),
    END_DATE
)

for _, rate_row in latest_rates.iterrows():
    current_rate = float(rate_row["avg_rate"])

    for rate_date in fx_months:
        # Small realistic monthly movement
        monthly_change = rng.normal(
            loc=0,
            scale=0.025
        )

        current_rate = max(
            current_rate * (1 + monthly_change),
            0.000001
        )

        generated_fx_records.append({
            "original_currency": rate_row["original_currency"],
            "target_currency": rate_row["target_currency"],
            "rate_date": rate_date,
            "avg_rate": current_rate
        })

generated_fx_long = pd.DataFrame(generated_fx_records)

complete_fx_long = pd.concat(
    [
        long_fx[
            identifier_columns
            + ["rate_date", "avg_rate"]
        ],
        generated_fx_long
    ],
    ignore_index=True
)

complete_fx_long = (
    complete_fx_long
    .drop_duplicates(
        subset=[
            "original_currency",
            "target_currency",
            "rate_date"
        ],
        keep="last"
    )
    .sort_values(
        identifier_columns + ["rate_date"]
    )
)

complete_fx_wide = (
    complete_fx_long
    .pivot(
        index=identifier_columns,
        columns="rate_date",
        values="avg_rate"
    )
    .reset_index()
)

complete_fx_wide.columns = [
    column.strftime(DATE_FORMAT)
    if isinstance(column, pd.Timestamp)
    else column
    for column in complete_fx_wide.columns
]


# =============================================================================
# FORMAT DATES
# =============================================================================

for column in [
    "transaction_date",
    "transaction_last_modified_date",
    "expected_delivery_date"
]:
    new_transactions[column] = (
        pd.to_datetime(new_transactions[column])
        .dt.strftime(DATE_FORMAT)
    )

new_transaction_lines[
    "transaction_line_last_modified_date"
] = (
    pd.to_datetime(
        new_transaction_lines[
            "transaction_line_last_modified_date"
        ]
    )
    .dt.strftime(DATE_FORMAT)
)

new_sales_budget["budget_date"] = (
    pd.to_datetime(new_sales_budget["budget_date"])
    .dt.strftime(DATE_FORMAT)
)


# =============================================================================
# SAVE GENERATED FILES
# =============================================================================

save_csv(
    new_transactions,
    "transactions_extended.csv"
)

save_csv(
    new_transaction_lines,
    "transactionline_extended.csv"
)

save_csv(
    new_sales_budget,
    "sales_budget_extended.csv"
)

save_csv(
    complete_fx_wide,
    "fx_avg_rate_extended.csv"
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\nGeneration complete")
print(f"New transactions: {len(new_transactions):,}")
print(f"New transaction lines: {len(new_transaction_lines):,}")
print(f"Budget rows including existing data: {len(new_sales_budget):,}")
print(f"FX currency pairs: {len(complete_fx_wide):,}")

Created: D:\Nazeer Joseph Data Universe\Data Load\Netsuite ETL Full Project\data\generated_data\transactions_extended.csv | Rows: 350,000
Created: D:\Nazeer Joseph Data Universe\Data Load\Netsuite ETL Full Project\data\generated_data\transactionline_extended.csv | Rows: 700,000
Created: D:\Nazeer Joseph Data Universe\Data Load\Netsuite ETL Full Project\data\generated_data\sales_budget_extended.csv | Rows: 2,340
Created: D:\Nazeer Joseph Data Universe\Data Load\Netsuite ETL Full Project\data\generated_data\fx_avg_rate_extended.csv | Rows: 35

Generation complete
New transactions: 350,000
New transaction lines: 700,000
Budget rows including existing data: 2,340
FX currency pairs: 35
